In [6]:
import os

for dirname, _, filenames in os.walk('/kaggle/input/datasets/lakrasmriti/crop-recommendation'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/lakrasmriti/crop-recommendation/Crop_recommendation.csv


In [7]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/lakrasmriti/crop-recommendation/Crop_recommendation.csv')

print(df.head())

    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice
3  74  35  40    26.491096  80.158363  6.980401  242.864034  rice
4  78  42  42    20.130175  81.604873  7.628473  262.717340  rice


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
import pickle

X = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y = df['label']


label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Support Vector Machine (SVM)": SVC(),
    "XGBoost": XGBClassifier(eval_metric='mlogloss', random_state=42)
}

model_performances = {}

print(" Training & Evaluating Models ")
for name, model in models.items():
    if name in ["Logistic Regression", "K-Nearest Neighbors (KNN)", "Support Vector Machine (SVM)"]:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
    acc = accuracy_score(y_test, y_pred) * 100
    model_performances[name] = acc
    print(f"-> {name} Accuracy: {acc:.2f}%")

best_model_name = max(model_performances, key=model_performances.get)
print(f"\n Best Performing Model: {best_model_name} with {model_performances[best_model_name]:.2f}%\n")


best_model = models[best_model_name]
if best_model_name in ["Logistic Regression", "K-Nearest Neighbors (KNN)", "Support Vector Machine (SVM)"]:
    best_model.fit(scaler.transform(X), y_encoded)
else:
    best_model.fit(X, y_encoded)


model_data = {
    "model": best_model,
    "scaler": scaler,
    "label_encoder": label_encoder,
    "model_type": best_model_name
}

with open('best_crop_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print(" Model successfully trained and saved as 'best_crop_model.pkl'")

--- Training & Evaluating Models ---
-> Logistic Regression Accuracy: 97.27%
-> K-Nearest Neighbors (KNN) Accuracy: 97.95%
-> Decision Tree Accuracy: 97.95%
-> Random Forest Accuracy: 99.55%
-> Support Vector Machine (SVM) Accuracy: 98.41%
-> XGBoost Accuracy: 99.09%

🏆 Best Performing Model: Random Forest with 99.55%

✅ Model successfully trained and saved as 'best_crop_model.pkl'!


In [10]:
import numpy as np
import pickle

def recommend_crop(n, p, k, temperature, humidity, ph, rainfall):
    
    with open('best_crop_model.pkl', 'rb') as f:
        data = pickle.load(f)
        
    model = data["model"]
    scaler = data["scaler"]
    label_encoder = data["label_encoder"]
    model_type = data["model_type"]

    user_input = np.array([[n, p, k, temperature, humidity, ph, rainfall]])
    
    if model_type in ["Logistic Regression", "K-Nearest Neighbors (KNN)", "Support Vector Machine (SVM)"]:
        user_input = scaler.transform(user_input)
        
    prediction_encoded = model.predict(user_input)
    crop_name = label_encoder.inverse_transform(prediction_encoded)
    
    return crop_name[0]

predicted_crop = recommend_crop(40, 72, 78, 530.87, 65.0, 6.5, 102.9)

print(f" Test Successful! Recommended Crop: {predicted_crop.upper()}")

✨ Test Successful! Recommended Crop: **CHICKPEA**


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [11]:
from IPython.display import FileLink
FileLink('best_crop_model.pkl')

/kaggle/working/best_crop_model.pkl